In [ ]:
import math
import re
import drawsvg as draw
import numpy as np

# =====================================
# Inputs (all lengths in mm)
# =====================================
# --- geometry ---
width = 19.5            # footprint width (given)
trace_angle = 45.0      # grid-line angle from the footprint's long axis, degrees.
                        #   0  = runs parallel to the height (axial strain)
                        #   45 = torsion gauge (use -45 for the mirror-image gauge)
edge_offset = 0.6       # keep-out from the footprint edge
trace_width = 0.15      # W
trace_spacing = trace_width
min_run = 1.0           # shortest allowed straight run of the serpentine
height_round = 0.01     # solved height is rounded UP to this step
max_height = 500.0      # give up searching beyond this height
tails_at_top = True     # True: both tails exit the top edge; False: the bottom edge
draw_outline = True     # draw the single width x height rectangle
image_scale = 15        # preview only
kicad_margin = 1.0      # blank border (mm) around the footprint in the KiCad SVGs

# --- electrical (target applies to the ONE trace, tails included) ---
target_resistance = 6.0     # ohms, at T_amb
copper_thickness = 0.035    # T, 1 oz/ft^2 copper
resistivity = 1.72e-8       # rho, ohm*m at T_ref (copper)
alpha = 0.00393             # temperature coefficient of resistivity, 1/degC (copper)
T_ref = 25.0                # reference temperature in the formula, degC
T_amb = 25.0                # ambient temperature, degC

# =====================================

assert width > 0
assert edge_offset >= 0 and edge_offset < width / 2
assert target_resistance > 0 and copper_thickness > 0 and trace_width > 0

RHO_OHM_MM = resistivity * 1000.0  # ohm*m -> ohm*mm so everything stays in mm


def temp_factor(t_amb=None):
    t_amb = T_amb if t_amb is None else t_amb
    return 1 + alpha * (t_amb - T_ref)


def trace_resistance(length_mm, t_amb=None):
    """R = rho * L / (T * W) * (1 + alpha * (T_amb - 25))"""
    return RHO_OHM_MM * length_mm / (copper_thickness * trace_width) * temp_factor(t_amb)


def length_for_resistance(r_ohm, t_amb=None):
    """Inverse of trace_resistance: the trace length that gives r_ohm."""
    return r_ohm * copper_thickness * trace_width / (RHO_OHM_MM * temp_factor(t_amb))


def path_length(dstr):
    """Length of an M/L/A path string. Arcs are quarter circles at any rotation."""
    segs = dstr.split(" ")
    ops = [x[0] for x in segs]
    assert ops[0] == "M"
    assert "M" not in ops[1:]

    total_len = 0
    origin = [float(x) for x in segs[0][1:].split(",")]
    for seg in segs[1:]:
        nex = [float(x) for x in seg[1:].split(",")]
        if seg[0] == "L":
            total_len += math.hypot(nex[0] - origin[0], nex[1] - origin[1])
            origin = nex
        elif seg[0] == "A":
            assert nex[0] == nex[1], "Not a circle"
            chord = math.hypot(nex[5] - origin[0], nex[6] - origin[1])
            assert abs(chord - nex[0] * math.sqrt(2)) < 2e-3, "Not a quarter circle"
            total_len += math.pi * nex[0] / 2
            origin = [nex[5], nex[6]]
        else:
            assert False, seg[0]
    return total_len


def calc_length(obj):
    return path_length(vars(obj)["args"]["d"])


# ---------------------------------------------------------------
# Geometry constants
# ---------------------------------------------------------------
tw, ts = trace_width, trace_spacing
rad = tw / 2                      # corner radius of the U-turns
pitch = tw + ts                   # centre-to-centre distance of neighbouring runs
x_margin = 2 * edge_offset        # side margin of the serpentine region
x_lead = edge_offset + tw         # centre line of the vertical lead channels
assert x_margin >= x_lead + tw / 2 + ts - 1e-9, \
    "edge_offset is too small to fit the lead channel beside the serpentine; increase it"

# The serpentine is built in a local frame (v across the runs, u along them),
# then rotated into the footprint. psi = grid angle, +180 deg if tails go to the top.
psi = math.radians(trace_angle) + (math.pi if tails_at_top else 0.0)
ev = (math.cos(psi), math.sin(psi))        # local v axis in footprint coordinates
eu = (-math.sin(psi), math.cos(psi))       # local u axis (direction of the runs)
tail_dir = (-eu[0], -eu[1])                # direction the tails leave in
assert abs(tail_dir[1]) > 0.1, "tails would run along the top/bottom edge; use |angle| below ~84 deg"


class LayoutError(ValueError):
    pass


def build(h):
    """Full trace (as a segment list) for a footprint of height h."""
    rect = (x_margin, edge_offset, width - x_margin, h - edge_offset)   # metal must stay inside
    cx, cy = width / 2, h / 2

    def G(v, u):
        return (cx + v * ev[0] + u * eu[0], cy + v * ev[1] + u * eu[1])

    def u_int(v):
        """Range of u for which the point (v, u) lies inside rect."""
        lo, hi = -math.inf, math.inf
        for c0, cv, cu, a, b in ((cx, ev[0], eu[0], rect[0], rect[2]),
                                 (cy, ev[1], eu[1], rect[1], rect[3])):
            base = c0 + v * cv
            if abs(cu) < 1e-12:
                if not (a - 1e-12 <= base <= b + 1e-12):
                    return math.inf, -math.inf
            else:
                t1, t2 = (a - base) / cu, (b - base) / cu
                lo, hi = max(lo, min(t1, t2)), min(hi, max(t1, t2))
        return lo, hi

    def attempt(N):
        M = N // 2
        vs = [(k - (N - 1) / 2) * pitch for k in range(N)]
        ranges = {}
        for v in vs:
            for dv in (-rad, rad):
                lo_, hi_ = u_int(v + dv)
                if not lo_ <= hi_:
                    return None
                ranges[(v, dv)] = (lo_, hi_)
        lo = lambda v, dv: ranges[(v, dv)][0]
        hi = lambda v, dv: ranges[(v, dv)][1]

        # flat U-turn caps: pairs (0,1),(2,3).. at the far end, (1,2),(3,4).. at the tail end
        top = [min(hi(vs[2 * m], -rad), hi(vs[2 * m + 1], rad)) for m in range(M)]
        bot = [max(lo(vs[2 * m + 1], -rad), lo(vs[2 * m + 2], rad)) for m in range(M - 1)]
        t_first = max(lo(vs[0], -rad), lo(vs[0], rad))
        t_last = max(lo(vs[N - 1], -rad), lo(vs[N - 1], rad))

        # straight-run length of every column
        runs = [top[0] - tw - t_first]
        for k in range(1, N - 1):
            if k % 2 == 1:
                m = (k - 1) // 2
                runs.append((top[m] - tw) - (bot[m] + tw))
            else:
                runs.append((top[k // 2] - tw) - (bot[(k - 2) // 2] + tw))
        runs.append(top[M - 1] - tw - t_last)
        if min(runs) < min_run:
            return None

        # local segments, travelling: tail -> far cap -> tail-end cap -> ... -> tail
        loc = [("M", vs[0], t_first)]
        for k in range(N):
            if k % 2 == 0:
                m = k // 2
                loc += [("L", vs[k], top[m] - tw),
                        ("A", vs[k] + rad, top[m] - rad, False),
                        ("L", vs[k + 1] - rad, top[m] - rad),
                        ("A", vs[k + 1], top[m] - tw, False)]
            elif k == N - 1:
                loc.append(("L", vs[k], t_last))
            else:
                m = (k - 1) // 2
                loc += [("L", vs[k], bot[m] + tw),
                        ("A", vs[k] + rad, bot[m] + rad, True),
                        ("L", vs[k + 1] - rad, bot[m] + rad),
                        ("A", vs[k + 1], bot[m] + tw, True)]
        return vs, loc, t_first, t_last

    diag = math.hypot(rect[2] - rect[0], rect[3] - rect[1])
    got = None
    for N in range(2 * int(diag / pitch / 2) + 2, 1, -2):
        got = attempt(N)
        if got:
            break
    if not got:
        raise LayoutError(f"no serpentine fits at height {h:.2f} mm")
    vs, loc, t_first, t_last = got
    N = len(vs)

    segs = []
    for s in loc:
        x, y = G(s[1], s[2])
        segs.append((s[0], x, y) + s[3:])

    # tails: continue each end run straight out, then either reach the top (or bottom)
    # edge directly, or turn into a vertical lead channel beside the serpentine.
    y_exit = h + 2 * tw if tail_dir[1] > 0 else -2 * tw

    def route(P):
        t_top = (y_exit - P[1]) / tail_dir[1]
        if abs(tail_dir[0]) < 1e-9:
            return [P, (P[0], y_exit)], None
        side = "left" if tail_dir[0] < 0 else "right"
        x_ch = x_lead if side == "left" else width - x_lead
        t_ch = (x_ch - P[0]) / tail_dir[0]
        if t_ch < 0:
            raise LayoutError("lead channel is behind the tail")
        if t_top <= t_ch:
            return [P, (P[0] + tail_dir[0] * t_top, y_exit)], None
        return [P, (x_ch, P[1] + tail_dir[1] * t_ch), (x_ch, y_exit)], side

    lead_s, side_s = route(G(vs[0], t_first))
    lead_e, side_e = route(G(vs[N - 1], t_last))
    if side_s is not None and side_s == side_e:
        raise LayoutError("both tails would share one lead channel at this angle")

    head = [("M",) + lead_s[-1]] + [("L",) + p for p in reversed(lead_s[:-1])]
    tail = [("L",) + p for p in lead_e[1:]]
    return head + segs[1:] + tail, dict(columns=N, tail_start=lead_s[-1], tail_end=lead_e[-1])


def segs_length(segs):
    total, cur = 0.0, segs[0][1:3]
    for s in segs[1:]:
        total += math.dist(cur, s[1:3]) if s[0] == "L" else math.pi * rad / 2
        cur = s[1:3]
    return total


def to_path(segs, color="green"):
    p = draw.Path(stroke_width=tw, stroke=color, fill_opacity=0)
    for s in segs:
        if s[0] == "M":
            p.M(s[1], s[2])
        elif s[0] == "L":
            p.L(s[1], s[2])
        else:
            p.A(rad, rad, 0, False, s[3], s[1], s[2])
    return p


# ---------------------------------------------------------------
# Solve for the footprint height that gives the target resistance
# ---------------------------------------------------------------
# With an angled grid the length is no longer exactly linear in height (columns
# appear and disappear), so search upward: coarse steps, then refine to height_round.
len_target = length_for_resistance(target_resistance)


def try_len(h):
    try:
        return segs_length(build(h)[0])
    except LayoutError:
        return None


coarse = 0.5
h = 2 * edge_offset + tw + min_run
while True:
    L = try_len(h)
    if L is not None and L >= len_target:
        break
    h += coarse
    if h > max_height:
        raise ValueError(f"target {target_resistance} ohm needs more than max_height = {max_height} mm "
                         f"at this width and angle; raise max_height or widen the footprint.")

height = None
for k in range(math.ceil(max(h - coarse, 0) / height_round - 1e-9), math.ceil(h / height_round - 1e-9) + 1):
    hk = round(k * height_round, 6)
    L = try_len(hk)
    if L is not None and L >= len_target:
        height = hk
        break
if height is None:
    height = round(h, 6)

# ---------------------------------------------------------------
# Final geometry and report
# ---------------------------------------------------------------
segs, info = build(height)
path_trace = to_path(segs)
length_trace = calc_length(path_trace)
R_actual = trace_resistance(length_trace)
assert R_actual <= target_resistance * 1.01, (
    f"Smallest layout this width/angle allows already gives {R_actual:.3f} ohm, above the "
    f"{target_resistance} ohm target. Raise the target, narrow the trace, or reduce min_run.")

outline = draw.Path(stroke_width=trace_width, stroke="RED", fill_opacity=0)
outline.M(0, 0)
outline.L(width, 0)
outline.L(width, height)
outline.L(0, height)
outline.Z()

print(f"footprint        : {width:.2f} x {height:.2f} mm (width given, height solved)")
print(f"grid angle       : {trace_angle:.1f} deg from the long axis, {info['columns']} runs")
print(f"target           : {target_resistance:.4f} ohm at {T_amb:.0f} degC -> length {len_target:.2f} mm")
print(f"trace length     : {length_trace:.2f} mm (tails included)")
print(f"trace resistance : {R_actual:.4f} ohm ({(R_actual / target_resistance - 1) * 100:+.3f}% vs target)")
print(f"tail ends (x, y) : ({info['tail_start'][0]:.3f}, {info['tail_start'][1]:.3f}) and "
      f"({info['tail_end'][0]:.3f}, {info['tail_end'][1]:.3f})  [origin = lower-left corner of the footprint]")

# ---------------------------------------------------------------
# Preview
# ---------------------------------------------------------------
d = draw.Drawing(width + 20, height + 20, origin=(-10, -10), displayInline=False)
if draw_outline:
    d.draw(outline)
d.draw(path_trace)
d.save_svg('curve_verbose.svg')
d.rasterize()
d.set_pixel_scale(image_scale)
d


In [ ]:
# ---------------------------------------------------------------
# KiCad export: SVGs whose width/height are real millimetres
# ---------------------------------------------------------------
def save_svg_mm(drawing, filename, w_mm, h_mm):
    """Save an SVG whose width/height are in mm, so 1 viewBox unit = 1 mm."""
    drawing.save_svg(filename)
    s = open(filename).read()
    tag = re.search(r"<svg[^>]*>", s).group(0)
    assert "viewBox" in tag, "no viewBox found"
    new = re.sub(r'(?<![\w-])width="[^"]*"', f'width="{w_mm:.4f}mm"', tag, count=1)
    new = re.sub(r'(?<![\w-])height="[^"]*"', f'height="{h_mm:.4f}mm"', new, count=1)
    open(filename, "w").write(s.replace(tag, new, 1))


def kicad_drawing(item):
    m = kicad_margin
    k = draw.Drawing(width + 2 * m, height + 2 * m, origin=(-m, -m), displayInline=False)
    k.draw(item)
    return k


# Both files share the same canvas size, so they line up when imported.
canvas_w, canvas_h = width + 2 * kicad_margin, height + 2 * kicad_margin
save_svg_mm(kicad_drawing(path_trace), "traces_kicad.svg", canvas_w, canvas_h)
save_svg_mm(kicad_drawing(outline), "outline_kicad.svg", canvas_w, canvas_h)
print(f"wrote traces_kicad.svg and outline_kicad.svg ({canvas_w:.2f} x {canvas_h:.2f} mm canvas)")


In [ ]:
# Check cell: reads the exported SVGs back from disk and verifies them.
import xml.etree.ElementTree as ET


def read_svg(fn):
    root = ET.parse(fn).getroot()
    paths = [el.attrib["d"] for el in root.iter() if el.tag.endswith("path")]
    w_attr, h_attr = root.attrib["width"], root.attrib["height"]
    assert w_attr.endswith("mm") and h_attr.endswith("mm"), f"{fn}: width/height are not in mm"
    vb = [float(v) for v in re.split(r"[,\s]+", root.attrib["viewBox"].strip())]
    assert abs(float(w_attr[:-2]) - vb[2]) < 1e-3 and abs(float(h_attr[:-2]) - vb[3]) < 1e-3, \
        f"{fn}: mm size does not match viewBox (scale is not 1 unit = 1 mm)"
    return paths, (float(w_attr[:-2]), float(h_attr[:-2]))


def endpoints(dstr):
    """End point of every M/L/A segment."""
    pts = []
    for seg in dstr.split(" "):
        if seg[0] not in "MLA":
            continue
        nums = [float(x) for x in re.split(r"[,\s]+", seg[1:]) if x]
        pts.append((nums[-2], nums[-1]))
    return pts


def min_clearance(pts, gap=5, chunk=200):
    """Smallest centre-line distance between path pieces that are not neighbours.
    Arcs are replaced by their chords, which is the conservative direction."""
    P = np.array(pts, dtype=float)
    A, B = P[:-1], P[1:]
    n = len(A)

    def pt_seg(X, S, T):
        ST = T - S
        t = np.clip(np.sum((X - S) * ST, -1) / np.maximum(np.sum(ST * ST, -1), 1e-18), 0, 1)
        return np.linalg.norm(X - (S + t[..., None] * ST), axis=-1)

    def cross(o, a, b):
        return (a[..., 0] - o[..., 0]) * (b[..., 1] - o[..., 1]) - (a[..., 1] - o[..., 1]) * (b[..., 0] - o[..., 0])

    best = math.inf
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        Ai, Bi = A[i0:i1, None], B[i0:i1, None]
        Aj, Bj = A[None], B[None]
        dist = np.minimum.reduce([pt_seg(Ai, Aj, Bj), pt_seg(Bi, Aj, Bj),
                                  pt_seg(Aj, Ai, Bi), pt_seg(Bj, Ai, Bi)])
        crossing = (cross(Aj, Bj, Ai) * cross(Aj, Bj, Bi) < 0) & (cross(Ai, Bi, Aj) * cross(Ai, Bi, Bj) < 0)
        dist = np.where(crossing, 0.0, dist)
        mask = (np.arange(n)[None, :] - np.arange(i0, i1)[:, None]) >= gap
        if mask.any():
            best = min(best, dist[mask].min())
    return best


# ---- traces ----
paths, (w_mm, h_mm) = read_svg("traces_kicad.svg")
assert len(paths) == 1, f"expected 1 trace path, got {len(paths)}"
assert abs(w_mm - (width + 2 * kicad_margin)) < 1e-3 and abs(h_mm - (height + 2 * kicad_margin)) < 1e-3
pts = endpoints(paths[0])

tail_y = height + 2 * trace_width if tails_at_top else -2 * trace_width
assert abs(pts[0][1] - tail_y) < 1e-9 and abs(pts[-1][1] - tail_y) < 1e-9, "tails are not on the expected edge"

lo = edge_offset if tails_at_top else -2 * trace_width
hi = height + 2 * trace_width if tails_at_top else height - edge_offset
for x, y in pts:
    assert edge_offset - 1e-9 <= x <= width - edge_offset + 1e-9, f"x={x} outside keep-out"
    assert lo - 1e-9 <= y <= hi + 1e-9, f"y={y} outside allowed band [{lo}, {hi}]"

clear = min_clearance(pts)
assert clear >= trace_width + trace_spacing - 1e-6, \
    f"traces come too close: centre-line distance {clear:.4f} mm < {trace_width + trace_spacing:.4f} mm"
print(f"min trace-to-trace spacing: {clear - trace_width:.4f} mm (required {trace_spacing:.4f} mm)")

r_svg = trace_resistance(path_length(paths[0]))
assert target_resistance - 1e-9 <= r_svg <= target_resistance * 1.01, \
    f"R={r_svg:.4f} ohm is not within 0..+1% of target {target_resistance} ohm"
print(f"trace: length {path_length(paths[0]):.2f} mm -> {r_svg:.4f} ohm")

# ---- outline ----
paths_o, (w_o, h_o) = read_svg("outline_kicad.svg")
assert len(paths_o) == 1, f"expected 1 outline path, got {len(paths_o)}"
assert (w_o, h_o) == (w_mm, h_mm), "outline and traces canvases differ, they won't line up"
opts = endpoints(paths_o[0])
ox, oy = [p[0] for p in opts], [p[1] for p in opts]
assert abs(min(ox)) < 1e-9 and abs(max(ox) - width) < 1e-9, "outline width is wrong"
assert abs(min(oy)) < 1e-9 and abs(max(oy) - height) < 1e-9, "outline height is wrong"
print(f"outline: {max(ox) - min(ox):.2f} x {max(oy) - min(oy):.2f} mm")

print("KiCad SVGs OK")
